In [1]:
pip install -U pandas requests google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompa

In [2]:
import time
import requests
import pandas as pd
import os
from google_play_scraper import reviews, Sort

In [ ]:
# Path to save scraped data
DATA_DIR = os.path.join(os.path.abspath(""), "data_new_approach")
os.makedirs(DATA_DIR, exist_ok=True)

APPS = [
    {"bank": "Chase",                    "ios_app_id": 298867247,  "gp_package": "com.chase.sig.android"},
    {"bank": "Bank of America",           "ios_app_id": 331323530,  "gp_package": "com.infonow.bofa"},
    {"bank": "Citi",                      "ios_app_id": 301460193,  "gp_package": "com.citi.citimobile"},
    {"bank": "Marcus by Goldman Sachs",   "ios_app_id": 1209168938, "gp_package": "com.goldman.savings.us.marcus"},
    {"bank": "Wells Fargo",               "ios_app_id": 207573461,  "gp_package": "com.wf.wellsfargomobile"},
]

In [4]:
def fetch_app_store_reviews_us(app_id: int, max_pages: int = 10, sleep_s: float = 0.8):
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
    })

    rows = []

    for page in range(1, max_pages + 1):
        url = f"https://itunes.apple.com/us/rss/customerreviews/page={page}/id={app_id}/sortby=mostrecent/json"
        r = s.get(url, timeout=30)

        ct = (r.headers.get("content-type") or "").lower()
        if r.status_code != 200:
            print(f"[Apple] app_id={app_id} stopped at page={page}, status={r.status_code}")
            break

        if "json" not in ct and "javascript" not in ct:
            print(f"[Apple] app_id={app_id} invalid content at page={page}")
            break

        data = r.json()
        entries = data.get("feed", {}).get("entry", [])

        for e in entries[1:]:  # skip app metadata
            rows.append({
                "platform": "app_store",
                "storefront": "us",
                "app_id": app_id,
                "review_id": e.get("id", {}).get("label"),
                "date": e.get("updated", {}).get("label"),
                "user": e.get("author", {}).get("name", {}).get("label"),
                "rating": int(e.get("im:rating", {}).get("label", 0)),
                "title": e.get("title", {}).get("label"),
                "review": e.get("content", {}).get("label"),
                "version": e.get("im:version", {}).get("label"),
            })

        time.sleep(sleep_s)

    return pd.DataFrame(rows)

In [5]:
def fetch_google_play_reviews_us(package: str, target: int = 500, sleep_s: float = 0.4):
    all_rows = []
    token = None

    while len(all_rows) < target:
        batch = min(200, target - len(all_rows))

        result, token = reviews(
            package,
            lang="en",
            country="us",
            sort=Sort.NEWEST,
            count=batch,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append({
                "platform": "google_play",
                "storefront": "us",
                "package": package,
                "review_id": r.get("reviewId"),
                "date": r.get("at"),
                "user": r.get("userName"),
                "rating": r.get("score"),
                "title": r.get("title"),
                "review": r.get("content"),
                "thumbsUpCount": r.get("thumbsUpCount"),
                "appVersion": r.get("reviewCreatedVersion"),
            })

        if token is None:
            break

        time.sleep(sleep_s)

    return pd.DataFrame(all_rows)

In [6]:
for app in APPS:
    bank = app["bank"]
    ios_id = app["ios_app_id"]
    pkg = app["gp_package"]

    print(f"\n Processing: {bank}")

    # App Store
    df_ios = fetch_app_store_reviews_us(ios_id)
    ios_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__app_store_us.csv")
    df_ios.to_csv(ios_path, index=False)
    print(f" App Store saved: {ios_path} ({len(df_ios)} rows)")

    # Google Play
    df_gp = fetch_google_play_reviews_us(pkg)
    gp_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__google_play_us.csv")
    df_gp.to_csv(gp_path, index=False)
    print(f" Google Play saved: {gp_path} ({len(df_gp)} rows)")

    # Combined
    df_all = pd.concat([df_ios, df_gp], ignore_index=True)
    combined_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__combined_us.csv")
    df_all.to_csv(combined_path, index=False)
    print(f"Combined saved: {combined_path}")


 Processing: Chase
 App Store saved: /content/data/chase__app_store_us.csv (490 rows)
 Google Play saved: /content/data/chase__google_play_us.csv (600 rows)
Combined saved: /content/data/chase__combined_us.csv

 Processing: Bank of America


KeyboardInterrupt: 